# Model Selection Analysis

Compare SLM models (8-30B) on vanilla LLaMEA to select the best model for the feasibility study.

**Criteria** (in priority order):
1. Failure rate (lower is better)
2. Best AOCC achieved
3. LLM inference time per candidate
4. Failure mode distribution

In [ ]:
import json
import re
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS_DIR = Path("../results_model_selection")

# Auto-detect which models have results
model_dirs = sorted([d for d in RESULTS_DIR.iterdir() if d.is_dir()]) if RESULTS_DIR.exists() else []
print(f"Found {len(model_dirs)} model(s): {[d.name for d in model_dirs]}")

In [ ]:
def parse_fitness(val):
    """Convert fitness from JSON (may be string '-inf') to float."""
    try:
        return float(val)
    except (TypeError, ValueError):
        return float("-inf")


def categorize_failure(entry):
    """Categorize a failed candidate into a root cause."""
    code = entry.get("code", "")
    feedback = entry.get("feedback", "")
    
    if not code or not code.strip():
        return "no_code"
    if "class " not in code:
        return "no_class"
    if "unexpected keyword argument 'budget'" in feedback:
        return "init_missing_budget"
    if "unexpected keyword argument 'dim'" in feedback:
        return "init_missing_dim"
    if "positional argument" in feedback and "bounds" in feedback:
        return "call_expects_bounds"
    if "__init__" in feedback and "argument" in feedback:
        return "init_signature_other"
    if "__call__" in feedback and "argument" in feedback:
        return "call_signature_other"
    if "import" in feedback.lower() or "ModuleNotFoundError" in feedback:
        return "import_error"
    if "SyntaxError" in feedback or "IndentationError" in feedback:
        return "syntax_error"
    return "runtime_error"


def load_model_data(model_dir):
    """Load log.jsonl and conversationlog.jsonl for a model."""
    # Find the run directory
    run_dirs = [d for d in model_dir.iterdir() if d.is_dir() and d.name.startswith("run-")]
    if not run_dirs:
        return None
    run_dir = run_dirs[0]
    
    log_file = run_dir / "log.jsonl"
    conv_file = run_dir / "conversationlog.jsonl"
    
    if not log_file.exists():
        return None
    
    # Load candidate log
    entries = []
    with open(log_file) as f:
        for line in f:
            entries.append(json.loads(line))
    
    # Load conversation timestamps
    conv_entries = []
    if conv_file.exists():
        with open(conv_file) as f:
            for line in f:
                conv_entries.append(json.loads(line))
    
    return {"entries": entries, "conv": conv_entries, "model": model_dir.name}


# Load all model data
all_data = {}
for d in model_dirs:
    data = load_model_data(d)
    if data:
        all_data[d.name] = data
        print(f"  {d.name}: {len(data['entries'])} candidates, {len(data['conv'])} conversation entries")

print(f"\nLoaded {len(all_data)} model(s)")

In [ ]:
# === Summary table ===
summary_rows = []

for tag, data in all_data.items():
    entries = data["entries"]
    conv = data["conv"]
    
    n_total = len(entries)
    fitnesses = [parse_fitness(e["fitness"]) for e in entries]
    n_fail = sum(1 for f in fitnesses if f == float("-inf"))
    n_success = n_total - n_fail
    valid_fitnesses = [f for f in fitnesses if f != float("-inf")]
    
    best_aocc = max(valid_fitnesses) if valid_fitnesses else float("nan")
    mean_aocc = np.mean(valid_fitnesses) if valid_fitnesses else float("nan")
    
    # Timing from conversation log
    total_time_h = float("nan")
    avg_llm_s = float("nan")
    avg_eval_success_s = float("nan")
    
    if conv:
        times = [datetime.strptime(e["time"], "%Y-%m-%d %H:%M:%S.%f") for e in conv]
        total_time_h = (times[-1] - times[0]).total_seconds() / 3600
        
        llm_times = []
        eval_success_times = []
        eval_fail_times = []
        gen_idx = 0
        
        for i in range(len(conv) - 1):
            t_curr = times[i]
            t_next = times[i + 1]
            delta = (t_next - t_curr).total_seconds()
            
            if conv[i]["role"] == "client" and conv[i+1]["role"] != "client":
                llm_times.append(delta)
            elif conv[i]["role"] != "client" and conv[i+1]["role"] == "client":
                if gen_idx < len(entries):
                    if fitnesses[gen_idx] == float("-inf"):
                        eval_fail_times.append(delta)
                    else:
                        eval_success_times.append(delta)
                    gen_idx += 1
        
        avg_llm_s = np.mean(llm_times) if llm_times else float("nan")
        avg_eval_success_s = np.mean(eval_success_times) if eval_success_times else float("nan")
    
    # Failure categorization
    failure_cats = {}
    for e, f in zip(entries, fitnesses):
        if f == float("-inf"):
            cat = categorize_failure(e)
            failure_cats[cat] = failure_cats.get(cat, 0) + 1
    
    summary_rows.append({
        "model": tag,
        "candidates": n_total,
        "success": n_success,
        "fail": n_fail,
        "fail_rate": n_fail / n_total * 100 if n_total else 0,
        "best_aocc": best_aocc,
        "mean_aocc": mean_aocc,
        "total_h": total_time_h,
        "avg_llm_s": avg_llm_s,
        "avg_eval_s": avg_eval_success_s,
        "failure_cats": failure_cats,
    })

df_summary = pd.DataFrame(summary_rows).set_index("model")
display_cols = ["candidates", "success", "fail", "fail_rate", "best_aocc", "mean_aocc",
                "total_h", "avg_llm_s", "avg_eval_s"]
df_summary[display_cols].round(2)

In [ ]:
# === Failure rate comparison ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = df_summary.index.tolist()
fail_rates = df_summary["fail_rate"].values
best_aoccs = df_summary["best_aocc"].values

colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(models)))

# Failure rate bar chart
ax = axes[0]
bars = ax.barh(models, fail_rates, color=colors)
ax.set_xlabel("Failure Rate (%)")
ax.set_title("Failure Rate by Model")
ax.set_xlim(0, 100)
for bar, rate in zip(bars, fail_rates):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f"{rate:.0f}%", va="center")

# Best AOCC bar chart
ax = axes[1]
bars = ax.barh(models, best_aoccs, color=colors)
ax.set_xlabel("Best AOCC")
ax.set_title("Best AOCC Achieved")
ax.set_xlim(0, 1)
for bar, aocc in zip(bars, best_aoccs):
    if not np.isnan(aocc):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f"{aocc:.3f}", va="center")

plt.tight_layout()
plt.savefig("model_selection_overview.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# === Failure mode breakdown per model ===
all_cats = set()
for row in summary_rows:
    all_cats.update(row["failure_cats"].keys())
all_cats = sorted(all_cats)

cat_data = {cat: [] for cat in all_cats}
for row in summary_rows:
    for cat in all_cats:
        cat_data[cat].append(row["failure_cats"].get(cat, 0))

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(models))
width = 0.7
bottom = np.zeros(len(models))

cat_colors = plt.cm.Set3(np.linspace(0, 1, len(all_cats)))
for i, cat in enumerate(all_cats):
    values = np.array(cat_data[cat])
    ax.bar(x, values, width, bottom=bottom, label=cat, color=cat_colors[i])
    bottom += values

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha="right")
ax.set_ylabel("Number of failures")
ax.set_title("Failure Mode Breakdown by Model")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("model_selection_failures.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# === Timing comparison ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LLM inference time
ax = axes[0]
llm_times = df_summary["avg_llm_s"].values
bars = ax.barh(models, llm_times, color=colors)
ax.set_xlabel("Avg LLM Inference Time (s)")
ax.set_title("LLM Response Time per Candidate")
for bar, t in zip(bars, llm_times):
    if not np.isnan(t):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f"{t:.0f}s", va="center")

# Total wall time
ax = axes[1]
total_h = df_summary["total_h"].values
bars = ax.barh(models, total_h, color=colors)
ax.set_xlabel("Total Wall Time (hours)")
ax.set_title("Total Experiment Duration")
for bar, t in zip(bars, total_h):
    if not np.isnan(t):
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                f"{t:.1f}h", va="center")

plt.tight_layout()
plt.savefig("model_selection_timing.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# === Convergence curves (best-so-far AOCC) ===
fig, ax = plt.subplots(figsize=(10, 6))

for tag, data in all_data.items():
    fitnesses = [parse_fitness(e["fitness"]) for e in data["entries"]]
    # Best-so-far (ignoring -inf)
    best_so_far = []
    current_best = float("-inf")
    for f in fitnesses:
        if f > current_best:
            current_best = f
        best_so_far.append(current_best if current_best != float("-inf") else np.nan)
    
    ax.plot(range(1, len(best_so_far) + 1), best_so_far, label=tag, linewidth=2)

ax.set_xlabel("Candidate #")
ax.set_ylabel("Best-so-far AOCC")
ax.set_title("Convergence: Best AOCC Over Candidates")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("model_selection_convergence.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# === Cost-efficiency: AOCC per compute hour ===
fig, ax = plt.subplots(figsize=(10, 6))

for i, (idx, row) in enumerate(df_summary.iterrows()):
    if np.isnan(row["best_aocc"]) or np.isnan(row["total_h"]):
        continue
    ax.scatter(row["total_h"], row["best_aocc"], s=200, color=colors[i],
               edgecolors="black", linewidth=1, zorder=5)
    ax.annotate(idx, (row["total_h"], row["best_aocc"]),
                textcoords="offset points", xytext=(10, 5), fontsize=9)

ax.set_xlabel("Total Wall Time (hours)")
ax.set_ylabel("Best AOCC")
ax.set_title("Cost-Efficiency: Best AOCC vs Compute Time")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("model_selection_efficiency.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# === Final ranking ===
# Composite score: low failure rate + high AOCC + reasonable speed

df_rank = df_summary[["fail_rate", "best_aocc", "mean_aocc", "avg_llm_s", "total_h"]].copy()

# Rank each metric (1 = best)
df_rank["rank_fail"] = df_rank["fail_rate"].rank(ascending=True)
df_rank["rank_aocc"] = df_rank["best_aocc"].rank(ascending=False)
df_rank["rank_speed"] = df_rank["avg_llm_s"].rank(ascending=True)

# Weighted composite (failure rate matters most)
df_rank["composite"] = (
    3 * df_rank["rank_fail"] +
    2 * df_rank["rank_aocc"] +
    1 * df_rank["rank_speed"]
)
df_rank["final_rank"] = df_rank["composite"].rank(ascending=True).astype(int)

print("=== FINAL MODEL RANKING ===")
print("Weights: failure_rate(3x) + best_aocc(2x) + speed(1x)")
print()
df_rank.sort_values("final_rank")[["fail_rate", "best_aocc", "avg_llm_s",
                                    "rank_fail", "rank_aocc", "rank_speed",
                                    "composite", "final_rank"]]